# Phase 3: Preprocessing & Data Pipeline

**Course:** Advanced AI Techniques (AAI)  
**Project:** AI-Enabled Micro-channel Battery Thermal Management System for Electric Vehicles (KJS-CES-02)  
**Dataset:** FreeTwinEV P45B 1s4p Battery Module Measurement  

This notebook demonstrates the end-to-end Phase 3 preprocessing and audit pipeline:
1. Locking the 14-run experimental design into Train (10), Validation (2), and Final Test (2)
2. The 3 Feature Experiments: Baseline (9 features), Extended (21 features), and Ablation (23 features)
3. Target formatting for 60s and 300s horizons with strict run-boundary terminal NaN dropping
4. Fitting RobustScaler strictly on the training partition
5. Automated Leakage Audit (PASS/FAIL verification)
6. Distribution and Scaling Audit (pre vs post transformation)
7. Distribution Shift Audit across the severe stress test runs (ID18, ID19)

**Strict Scientific Rule:** No ML models are trained in this phase.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import json
import joblib

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

prep_dir = project_root / 'results' / 'freetwinev' / 'preprocessed'
print('Preprocessing output directory exists:', prep_dir.exists())

## 1. Experimental Split Manifest

The run-level split assigns 10 runs to Train, 2 to Validation, and 2 to Final Test.  
No observations from the same run appear across multiple splits.

In [ ]:
df_split = pd.read_csv(prep_dir / 'split_manifest.csv')
print(f"Total runs: {len(df_split)}")
print("Split counts:", df_split['Split'].value_counts().to_dict())
df_split[['Exp_ID', 'Split', 'Current_Load', 'Coolant_Temp', 'Flow_Rate', 'Raw_Rows', 'Valid_60s_Rows', 'Valid_300s_Rows']]

## 2. Feature Experiment Manifests

Overview of Baseline (9), Extended (21), and Ablation (23) feature sets.

In [ ]:
df_feat_man = pd.read_csv(prep_dir / 'feature_manifest.csv')
print(f"Total features in taxonomy: {len(df_feat_man)}")
df_feat_man[['Feature', 'Category', 'Unit', 'Availability', 'Window', 'Leakage_Status']]

## 3. Automated Preprocessing Leakage Audit

Verifies that no target, future signal, or test statistic contaminated the pipeline.

In [ ]:
with open(prep_dir / 'leakage_audit_results.json', 'r') as f:
    audit_res = json.load(f)

for exp_key, result in audit_res.items():
    print(f"Experiment {exp_key}: Status = {result['status']}")
    for chk, details in result['checks'].items():
        print(f"   - {chk}: Passed = {details['passed']}")

## 4. Scaling Verification (Pre vs Post RobustScaler)

Verifies that training features achieve median = 0 and IQR = 1, while validation and test are transformed using training statistics.

In [ ]:
df_scaling = pd.read_csv(prep_dir / 'scaling_audit.csv')
df_scaling.head(10)

## 5. Distribution Shift Analysis on Severe Stress Tests (ID18, ID19)

Highlights features where test conditions intentionally exceed training boundaries due to starved cooling.

In [ ]:
df_shift = pd.read_csv(prep_dir / 'distribution_shift_audit.csv')
shifted = df_shift[df_shift['Shift_Detected']]
print(f"Features exhibiting distribution shift in test set: {len(shifted)}")
shifted[['Experiment', 'Feature', 'Train_Range', 'Test_Range', 'Shift_Description']].head(10)

## 6. Preprocessed Datasets Dimension Summary

In [ ]:
with open(prep_dir / 'preprocessing_config.json', 'r') as f:
    config = json.load(f)
print("Preprocessing Configuration Summary:")
print(json.dumps(config, indent=2))